In [1]:
import osmnx as ox
import networkx as nx
import numpy as np
import pandas as pd
from tqdm import tqdm
import time
from scipy import linalg
import porespy as ps
from skimage import draw
import pickle
import os
import glob
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

In [2]:
csv_data = []

# Function to create a 2D binary image from road network

def graph_to_binary_image(G, image_size=700):
    """Convert road network graph to a 2D binary image with enhanced clarity."""
    coords = np.array([(data['x'], data['y']) for node, data in G.nodes(data=True)])
    x_min, x_max = coords[:, 0].min(), coords[:, 0].max()
    y_min, y_max = coords[:, 1].min(), coords[:, 1].max()
    extent = max(x_max - x_min, y_max - y_min) * 1.1
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    im = np.zeros((image_size, image_size), dtype=bool)
   
    def scale_coord(coord, center, extent):
        return int(((coord - center) / extent + 0.5) * (image_size - 1))
    for u, v, data in G.edges(data=True):
        if 'geometry' in data:
            xs, ys = data['geometry'].xy
            coords = np.array(list(zip(xs, ys)))
        else:
            x1, y1 = G.nodes[u]['x'], G.nodes[u]['y']
            x2, y2 = G.nodes[v]['x'], G.nodes[v]['y']
            coords = np.array([[x1, y1], [x2, y2]])
   
        for i in range(len(coords) - 1):
            x0, y0 = scale_coord(coords[i, 0], x_center, extent), scale_coord(coords[i, 1], y_center, extent)
            x1, y1 = scale_coord(coords[i + 1, 0], x_center, extent), scale_coord(coords[i + 1, 1], y_center, extent)
            rr, cc = draw.line(y0, x0, y1, x1)
            for r, c in zip(rr, cc):
                if 0 <= r < image_size and 0 <= c < image_size:
                    rr_disk, cc_disk = draw.disk((r, c), radius=2, shape=(image_size, image_size))
                    im[rr_disk, cc_disk] = True
    return im
    
def get_edge_coords(G, u, v):
    data = G.get_edge_data(u, v)
    if not data:
        return np.array([])
    key = list(data.keys())[0]
    data = data[key]
    if 'geometry' in data:
        xs, ys = data['geometry'].xy
        coords = np.array(list(zip(xs, ys)))
    else:
        coords = np.array([[G.nodes[u]['x'], G.nodes[u]['y']], [G.nodes[v]['x'], G.nodes[v]['y']]])
    u_pos = np.array([G.nodes[u]['x'], G.nodes[u]['y']])
    dist_start = np.linalg.norm(coords[0] - u_pos)
    dist_end = np.linalg.norm(coords[-1] - u_pos)
    if dist_start > dist_end:
        coords = coords[::-1]
    return coords
    
def compute_branches(G):
    """Compute lengths and polylines of branches in the road network."""
    adj = {n: list(G.neighbors(n)) for n in G}
    degrees = {n: len(adj[n]) for n in adj}
    key_nodes = set(n for n in degrees if degrees[n] != 2)
    branch_lengths = []
    branch_polylines = []
    visited = set()
    for kn in key_nodes:
        for nb in adj[kn]:
            edge = frozenset([kn, nb])
            if edge in visited:
                continue
            visited.add(edge)
            # Start poly with coords of this edge
            coords = get_edge_coords(G, kn, nb)
            if len(coords) == 0:
                continue
            poly = coords.tolist()
            # Initial segment length
            edge_data = G.get_edge_data(kn, nb)
            key = list(edge_data.keys())[0]
            seg_len = edge_data[key]['length']
            added_length = seg_len
            # Now trace from nb, prev=kn
            current = nb
            prev = kn
            while True:
                neighbors = [nxt for nxt in adj[current] if nxt != prev]
                if len(neighbors) != 1:
                    break
                nxt = neighbors[0]
                edge = frozenset([current, nxt])
                if edge in visited:
                    break
                visited.add(edge)
                edge_data = G.get_edge_data(current, nxt)
                if not edge_data:
                    break
                key = list(edge_data.keys())[0]
                edge_len = edge_data[key]['length']
                added_length += edge_len
                # Append coords
                new_coords = get_edge_coords(G, current, nxt)
                if len(new_coords) == 0:
                    break
                if np.allclose(poly[-1], new_coords[0], atol=1e-6):
                    poly.extend(new_coords[1:].tolist())
                else:
                    poly.extend(new_coords.tolist())
                prev = current
                current = nxt
            branch_len = added_length
            if branch_len > 0:
                branch_lengths.append(branch_len)
                branch_polylines.append(np.array(poly))
    return np.array(branch_lengths), branch_polylines
    
def average_bending_energy_curve(c, closed=False, eps=1e-12):
    """
    Compute the line integral of the squared curvature of a polyline:
        ∫ κ(s)^2 ds / len(c)
    using a discrete arclength-based scheme on edges/tangents.
    Parameters
    ----------
    c : (N, d) array_like
        Vertices of the curve (d=2 or d=3).
    closed : bool, default True
        If True, treat the curve as closed (cyclic indices).
        If False, treat as open and use only interior vertices.
    eps : float, default 1e-12
        Small clamp to avoid division by zero for tiny edges.
    Returns
    -------
    float
        Discrete approximation of ∫ κ^2 ds.
    Notes
    -----
    Let e_i = c_{i+1} - c_i, ℓ_i = ||e_i||, and t_{i+1/2} = e_i / ℓ_i.
    For a closed curve, the per-vertex contribution at vertex i is:
        ||t_{i+1/2} - t_{i-1/2}||^2 / ((ℓ_i + ℓ_{i-1})/2).
    For an open curve, the sum is taken over interior vertices only.
    """
    c = np.asarray(c, dtype=np.float64)
    if c.ndim != 2 or c.shape[1] < 2:
        raise ValueError("c must be an (N, d) array with d >= 2.")
    N, d = c.shape
    if N < 3:
        raise ValueError("Need at least 3 points to estimate curvature.")
    if closed:
        # Edges, lengths, and unit tangents on edges
        e = np.roll(c, -1, axis=0) - c # (N, d), edges i -> i+1
        ℓ = np.linalg.norm(e, axis=1)
        length = np.sum(ℓ)
        ℓ = np.maximum(ℓ, eps)
        t = e / ℓ[:, None] # unit tangents on edges
        # Δt at vertices (i): t_{i+1/2} - t_{i-1/2}
        Δt = t - np.roll(t, 1, axis=0) # (N, d)
        # Dual (Voronoi) cell length around vertex i
      
        avg_ℓ = 0.5 * (ℓ + np.roll(ℓ, 1)) # (N,)
        avg_ℓ = np.maximum(avg_ℓ, eps)
        # Sum of ||Δt||^2 / avg_ℓ
        energy = np.sum(np.einsum('ij,ij->i', Δt, Δt) / avg_ℓ)/length
        return float(energy)
    else:
        # Open curve: edges on [0..N-2]
        e = c[1:] - c[:-1] # (N-1, d)
        ℓ = np.linalg.norm(e, axis=1)
        length = np.sum(ℓ)
        ℓ = np.maximum(ℓ, eps)
        t = e / ℓ[:, None] # (N-1, d)
        # Interior vertices correspond to differences of edge tangents
        Δt = t[1:] - t[:-1] # (N-2, d)
        avg_ℓ = 0.5 * (ℓ[1:] + ℓ[:-1]) # (N-2,)
        avg_ℓ = np.maximum(avg_ℓ, eps)
        energy = np.sum(np.einsum('ij,ij->i', Δt, Δt) / avg_ℓ)/length
        return float(energy)
        
def compute_fractal_dimension(G):
    """Compute fractal dimension using box-counting on a 2D binary image in specified range."""
    try:
        binary_image = graph_to_binary_image(G, image_size=700)
        box_sizes = np.logspace(np.log10(5), np.log10(50), num=5, dtype=int)
        boxcount = ps.metrics.boxcount(binary_image, bins=box_sizes)
        fds = boxcount.slope
        return np.mean(fds) if len(fds) > 0 else 0.0
    except Exception as e:
        print(f"Fractal dimension calculation error: {e}")
        return 0.0
       

pickle_folder = '../pkl/pkl_city_graphs'

pickle_files = glob.glob(os.path.join(pickle_folder, '*.pkl'))
for pickle_file in tqdm(pickle_files):
    print(f"\nProcessing pickle file: {pickle_file}")
  
    try:
        with open(pickle_file, 'rb') as f:
            combined_graph = pickle.load(f)
      
        # Extract name and type from graph attributes
        name = combined_graph.graph.get('name', os.path.basename(pickle_file).replace('.pkl', '').replace('_', ' '))
        region_type = combined_graph.graph.get('type', 'unknown')
      
        print(f"Loaded graph for {name} (type: {region_type})")
    except Exception as e:
        print(f"Error loading {pickle_file}: {e}")
        continue
  
    # Compute branches
    branch_lengths, branch_polylines = compute_branches(combined_graph)
    number_of_egdes = len(branch_lengths)
    avg_branch_length = np.mean(branch_lengths) if branch_lengths.size > 0 else 0
    
    # Compute edge directions and weights
    local_vectors = []
    local_lengths = []
    straight_line_lengths = []
    all_points = [] 


    for u, v, data in combined_graph.edges(data=True):
        if 'geometry' in data:
            xs, ys = data['geometry'].xy
            coords = np.array(list(zip(xs, ys)))
        else:
            x1, y1 = combined_graph.nodes[u]['x'], combined_graph.nodes[u]['y']
            x2, y2 = combined_graph.nodes[v]['x'], combined_graph.nodes[v]['y']
            coords = np.array([[x1, y1], [x2, y2]])

        # accumulate points for convex hull
        all_points.extend(coords.tolist())  # <- safer: convert array rows to list

        total_edge_length = 0
        for i in range(len(coords) - 1):
            x0, y0 = coords[i]
            x1, y1 = coords[i + 1]
            dx, dy = x1 - x0, y1 - y0
            length = np.sqrt(dx**2 + dy**2)
            if length > 0:
                local_vectors.append((dx / length, dy / length))
                local_lengths.append(length)
                total_edge_length += length

        # Straight-line distance between start and end nodes
        straight_distance = np.sqrt((coords[-1, 0] - coords[0, 0])**2 +
                                    (coords[-1, 1] - coords[0, 1])**2)
        straight_line_lengths.append(straight_distance)

    # Skip if no valid segments
    if len(local_vectors) == 0:
        print(f"No valid segments for {name}")
        continue

    # ---- edge density: length / area ----
    all_points = np.array(all_points)
    if len(np.unique(all_points, axis=0)) >= 3:
        hull = ConvexHull(all_points)
        area = hull.volume  # in 2D this is area
    else:
        area = 0
        print(f"Warning: Not enough unique points to compute convex hull for {name}. Setting area to 0.")

    total_path_length = sum(local_lengths)
    branch_density = total_path_length / area if area > 0 else 0
    # -----------------------------------------


    
    try:
        bc = nx.betweenness_centrality(combined_graph, weight='length', normalized=True)


        
        mean_betweenness_centrality = np.mean(list(bc.values())) if bc else 0
    except:
        mean_betweenness_centrality = 0
    try:
        graph_diameter = nx.diameter(combined_graph, weight='length') if nx.is_connected(combined_graph) else 0
    except:
        graph_diameter = 0
    try:
        average_path_length = nx.average_shortest_path_length(combined_graph, weight='length') if nx.is_connected(combined_graph) else 0
    except:
        average_path_length = 0
    try:
        assortativity = nx.degree_assortativity_coefficient(combined_graph, weight='length') if combined_graph.number_of_edges() >= 2 else 0
        assortativity = 0 if np.isnan(assortativity) else assortativity
    except:
        assortativity = 0
    try:
        L = nx.laplacian_matrix(combined_graph, weight='length').astype(float).toarray()
        eigenvalues = linalg.eigvalsh(L)
        eigenvalues = np.maximum(eigenvalues, 1e-12)
        probs = eigenvalues / np.sum(eigenvalues)
        spectral_entropy = -np.sum(probs * np.log(probs)) if probs.sum() > 0 else 0
    except:
        spectral_entropy = 0
      
    algebraic_connectivity = nx.algebraic_connectivity(combined_graph, weight='length', method='tracemin_pcg') if nx.is_connected(combined_graph) else 0
  
    # Compute circuity
    total_path_length = sum(local_lengths)
    total_straight_length = sum(straight_line_lengths) if straight_line_lengths else 0
    circuity = total_path_length / total_straight_length if total_straight_length > 0 else 1.0
    
    # Compute angular metrics
    angles = np.arctan2([dy for dx, dy in local_vectors], [dx for dx, dy in local_vectors]) % np.pi
    angles_full = 2 * angles
    edge_weights = np.array(local_lengths) / np.sum(local_lengths)
    C_p = np.sum(edge_weights * np.cos(angles_full))
    S_p = np.sum(edge_weights * np.sin(angles_full))
    theta_mean_full = np.arctan2(S_p, C_p) % (2 * np.pi)
  
    theta_mean = theta_mean_full / 2
    R_p_mean = np.sqrt(C_p**2 + S_p**2)
  
    directional_std_dev = np.sqrt(-2 * np.log(R_p_mean)) if R_p_mean > 0 else 0
    rotated_angles = (angles - theta_mean + np.pi/2) % np.pi
  
    mask_4 = ((rotated_angles >= 0) & (rotated_angles <= np.pi/8)) | ((rotated_angles >= 7*np.pi/8) & (rotated_angles <= np.pi))
    mask_3 = ((rotated_angles > np.pi/8) & (rotated_angles <= np.pi/4)) | ((rotated_angles >= 3*np.pi/4) & (rotated_angles < 7*np.pi/8))
    mask_2 = ((rotated_angles > np.pi/4) & (rotated_angles <= 3*np.pi/8)) | ((rotated_angles >= 5*np.pi/8) & (rotated_angles < 3*np.pi/4))
    mask_1 = (rotated_angles > 3*np.pi/8) & (rotated_angles < 5*np.pi/8)
  
    mass_quartile_1 = np.sum(edge_weights * mask_1)
    mass_quartile_2 = np.sum(edge_weights * mask_2)
    mass_quartile_3 = np.sum(edge_weights * mask_3)
    mass_quartile_4 = np.sum(edge_weights * mask_4)
  
    n_bins = 18
    bin_edges = np.linspace(0, np.pi, n_bins + 1)
    hist_weighted, _ = np.histogram(rotated_angles, bins=bin_edges, weights=edge_weights)
    p_weighted = hist_weighted / np.sum(hist_weighted) if np.sum(hist_weighted) > 0 else np.zeros_like(hist_weighted)
    directional_entropy = -np.sum([p * np.log(p) if p > 0 else 0 for p in p_weighted])
    orientation_order = 1 - (directional_entropy / np.log(18))**2 if directional_entropy > 0 else 0
    # Compute fractal dimension
    fractal_dimension = compute_fractal_dimension(combined_graph)
  
    # Compute average bending energy
    branch_avg_bendings = []
    for c in branch_polylines:
        if c.shape[0] >= 3:
            try:
                avg_bend = average_bending_energy_curve(c, closed=False)
                branch_avg_bendings.append(avg_bend)
            except ValueError as e:
                print(f"Skipping branch in {name}: {e}")
    bending_energy = sum(branch_avg_bendings) if branch_avg_bendings else 0.0
  
 # Prepare data for CSV
    csv_row = {
        'region_name': name,
        'type': region_type,
        
        
        'N_edg': number_of_egdes,           
        'Betw': mean_betweenness_centrality,
        'Spec_ent': spectral_entropy,
        'Alg_Conn': algebraic_connectivity,
        'Assort': assortativity,
        'G_diam': graph_diameter,
        'Br_dens': branch_density,           
        'Br_l': avg_branch_length,
        'Energy': bending_energy,
        'Apl': average_path_length,
        'Circ': circuity,
        'FD': fractal_dimension,
        'Dir_std': directional_std_dev,
        'q1': mass_quartile_1,
        'q2': mass_quartile_2,
        'q3': mass_quartile_3,
        'q4': mass_quartile_4,
        'Dir_ent': directional_entropy,
        'Order': orientation_order
    }
    csv_data.append(csv_row)
   

    
# Save features to CSV
csv_columns = [
    'region_name', 'type',
    'N_edg',
    'Betw',
    'Spec_ent',
    'Alg_Conn',
    'Assort',
    'G_diam',
    'Br_dens',
    'Br_l',
    'Energy',
    'Apl',
    'Circ',
    'FD',
    'Dir_std',
    'q1',
    'q2',
    'q3',
    'q4',
    'Dir_ent',
    'Order'
]

df = pd.DataFrame(csv_data, columns=csv_columns)
df.to_csv('main_road_network_features.csv', index=False)
print("\nFeature data saved to 'main_road_network_features.csv'")

  0%|                                                   | 0/100 [00:00<?, ?it/s]


Processing pickle file: ../pkl/pkl_city_graphs/Florence_Italy.pkl
Loaded graph for Florence, Italy (type: grid)


  0%|          | 0/5 [00:00<?, ?it/s]

  1%|▍                                          | 1/100 [00:01<01:54,  1.16s/it]


Processing pickle file: ../pkl/pkl_city_graphs/Old_Montreal_Quebec_Canada.pkl
Loaded graph for Old Montreal, Quebec, Canada (type: hybrid)


...

 99%|█████████████████████████████████████████▌| 99/100 [03:24<00:01,  1.05s/it]


Processing pickle file: ../pkl/pkl_city_graphs/Warsaw_Poland.pkl
Loaded graph for Warsaw, Poland (type: organic)


  0%|          | 0/5 [00:00<?, ?it/s]

100%|█████████████████████████████████████████| 100/100 [03:25<00:00,  2.05s/it]


Feature data saved to 'main_road_network_features.csv'
